# 🚀 Projeto G2 — Análise de Startups e Inovação Tecnológica no Brasil

## Tema 30

Este notebook apresenta uma análise exploratória, visual e interpretativa do ecossistema de startups e inovação tecnológica no Brasil entre 2015 e 2024.

O objetivo é investigar setores, regiões, estados, tecnologias, investimentos, faturamento e crescimento, gerando evidências para apoiar tomada de decisão.

## 1. Introdução e contextualização

Startups são empresas com modelo de negócio escalável, alto potencial de crescimento e forte relação com inovação tecnológica. No Brasil, esse ecossistema tem se expandido em setores como fintechs, healthtechs, edtechs, agritechs, inteligência artificial, blockchain, cloud e comércio digital.

Analisar dados sobre startups permite compreender onde a inovação está concentrada, quais setores estão recebendo mais investimento, quais regiões se destacam e se o investimento recebido está associado ao crescimento das empresas.

Este projeto segue uma lógica analítica profissional: leitura dos dados, limpeza, preparação, criação de KPIs, visualizações, interpretação e conclusão executiva.

## 2. Pergunta central do projeto

**Quais setores, regiões, estados e tecnologias concentram maior força inovadora no Brasil, e como investimento, faturamento e crescimento ajudam a explicar esse ecossistema?**

### Relevância da pergunta

Essa pergunta é relevante porque permite apoiar decisões de:
- investidores que desejam identificar setores promissores;
- gestores públicos que buscam fortalecer polos de inovação;
- incubadoras e aceleradoras que precisam selecionar áreas de maior potencial;
- empreendedores que desejam entender tendências do mercado.

### Decisão apoiada pela análise

Com base nos resultados, seria possível decidir onde priorizar investimentos, programas de aceleração, políticas públicas de incentivo e estratégias de expansão tecnológica.

## 3. Definição dos KPIs

Os KPIs escolhidos foram:

1. **Total de startups únicas** — mede o tamanho do ecossistema.
2. **Investimento total recebido** — mede o volume financeiro captado.
3. **Faturamento total estimado** — indica resultado econômico.
4. **Crescimento médio percentual** — avalia expansão média.
5. **Setor mais promissor** — setor com maior investimento total.
6. **Estado mais inovador** — estado com maior investimento total.
7. **Startup com maior faturamento** — destaque empresarial.

Esses indicadores foram escolhidos porque combinam quantidade, capital, desempenho, crescimento e distribuição geográfica.

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 4. Aquisição dos dados

A base utilizada é o arquivo `simulacao_startups_brasil.csv`, disponibilizado para o Projeto G2 — Tema 30.

O dataset contém informações simuladas sobre startups brasileiras, incluindo ano, mês, região, estado, cidade, setor, funcionários, investimento recebido, faturamento, crescimento percentual, estágio, tecnologia principal e nível de inovação.

In [ ]:
# Leitura da base de dados
df = pd.read_csv('../dados/simulacao_startups_brasil.csv')

# Visualização inicial
df.head()

In [ ]:
# Dimensão da base
print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')

df.info()

In [ ]:
# Verificação de valores ausentes
df.isna().sum()

In [ ]:
# Verificação de duplicidades
df.duplicated().sum()

## 5. Limpeza e preparação dos dados

Nesta etapa, serão feitas verificações de qualidade, conversão de tipos e criação de variáveis auxiliares.

Mesmo sendo uma base simulada e aparentemente limpa, a etapa de preparação é essencial para garantir consistência analítica.

In [ ]:
# Padronização de nomes de colunas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

# Conversão da coluna de data
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# Criação de variáveis auxiliares
df['ano_mes'] = df['data'].dt.to_period('M').astype(str)
df['investimento_milhoes'] = df['investimento_recebido'] / 1_000_000
df['faturamento_milhoes'] = df['faturamento'] / 1_000_000
df['faturamento_por_funcionario'] = df['faturamento'] / df['funcionarios']

df.head()

In [ ]:
# Validação após preparação
validacao = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'valores_unicos': df.nunique()
})

validacao

## 6. Análise exploratória inicial

A análise exploratória permite compreender a estrutura geral da base, a distribuição das variáveis numéricas e a frequência das categorias principais.

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Frequência das principais variáveis categóricas
for coluna in ['regiao', 'uf', 'setor', 'estagio', 'tecnologia_principal', 'nivel_inovacao']:
    print(f'\n--- {coluna.upper()} ---')
    print(df[coluna].value_counts())

## 7. KPIs gerais do ecossistema

In [ ]:
total_startups = df['startup'].nunique()
investimento_total = df['investimento_recebido'].sum()
faturamento_total = df['faturamento'].sum()
crescimento_medio = df['crescimento_percentual'].mean()
setor_mais_promissor = df.groupby('setor')['investimento_recebido'].sum().idxmax()
estado_mais_inovador = df.groupby('uf')['investimento_recebido'].sum().idxmax()
startup_maior_faturamento = df.groupby('startup')['faturamento'].sum().idxmax()

kpis = pd.DataFrame({
    'KPI': [
        'Total de startups únicas',
        'Investimento total recebido',
        'Faturamento total estimado',
        'Crescimento médio percentual',
        'Setor mais promissor',
        'Estado mais inovador',
        'Startup com maior faturamento'
    ],
    'Resultado': [
        total_startups,
        investimento_total,
        faturamento_total,
        crescimento_medio,
        setor_mais_promissor,
        estado_mais_inovador,
        startup_maior_faturamento
    ]
})

kpis

### Interpretação dos KPIs

Os KPIs mostram o tamanho e a força financeira do ecossistema analisado. O total de startups indica a diversidade de empresas, enquanto investimento e faturamento ajudam a medir maturidade econômica. O crescimento médio mostra a velocidade de expansão, e os rankings por setor, estado e startup destacam os principais polos de desempenho.

## 8. Evolução temporal do ecossistema

In [ ]:
temporal = df.groupby('ano').agg(
    startups=('startup', 'nunique'),
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean')
).reset_index()

temporal

In [ ]:
fig = px.line(
    temporal,
    x='ano',
    y='investimento_total',
    markers=True,
    title='Evolução anual do investimento recebido pelas startups',
    labels={'ano': 'Ano', 'investimento_total': 'Investimento recebido (R$)'}
)

fig.update_layout(template='plotly_white')
fig.show()

### Interpretação

A evolução temporal permite observar se o ecossistema está recebendo mais capital ao longo dos anos. Mesmo quando a quantidade de registros por ano é estável, o investimento pode variar, indicando ciclos de maior ou menor apetite do mercado por startups.

## 9. Comparação entre setores

In [ ]:
setores = df.groupby('setor').agg(
    quantidade_registros=('startup', 'count'),
    startups_unicas=('startup', 'nunique'),
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean'),
    funcionarios_medios=('funcionarios', 'mean')
).reset_index().sort_values('investimento_total', ascending=False)

setores

In [ ]:
fig = px.bar(
    setores,
    x='setor',
    y='investimento_total',
    title='Investimento total por setor',
    labels={'setor': 'Setor', 'investimento_total': 'Investimento recebido (R$)'}
)

fig.update_layout(template='plotly_white', xaxis_tickangle=-35)
fig.show()

### Interpretação

A comparação por setor identifica quais áreas tecnológicas concentram maior volume de capital. Setores com maior investimento tendem a ser mais atrativos para investidores, mas devem ser avaliados também pelo crescimento médio e pelo faturamento para evitar uma conclusão baseada apenas em volume financeiro.

## 10. Comparação regional e estadual

In [ ]:
regioes = df.groupby('regiao').agg(
    startups_unicas=('startup', 'nunique'),
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean')
).reset_index().sort_values('investimento_total', ascending=False)

regioes

In [ ]:
fig = px.bar(
    regioes,
    x='regiao',
    y=['investimento_total', 'faturamento_total'],
    barmode='group',
    title='Investimento e faturamento por região',
    labels={'regiao': 'Região', 'value': 'Valor (R$)', 'variable': 'Indicador'}
)

fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
estados = df.groupby('uf').agg(
    quantidade_registros=('startup', 'count'),
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean')
).reset_index().sort_values('investimento_total', ascending=False)

estados.head(15)

In [ ]:
fig = px.bar(
    estados.head(15),
    x='uf',
    y='investimento_total',
    title='Top 15 estados por investimento recebido',
    labels={'uf': 'Estado', 'investimento_total': 'Investimento recebido (R$)'}
)

fig.update_layout(template='plotly_white')
fig.show()

### Interpretação

A análise regional mostra onde o ecossistema de inovação está mais concentrado. Regiões e estados com maior investimento podem indicar presença de hubs tecnológicos, maior acesso a capital, universidades, aceleradoras, empresas de tecnologia e mão de obra qualificada.

## 11. Relação entre investimento e crescimento

In [ ]:
correlacao = df[['investimento_recebido', 'crescimento_percentual', 'faturamento', 'funcionarios']].corr()
correlacao

In [ ]:
fig = px.imshow(
    correlacao,
    text_auto=True,
    title='Mapa de correlação entre variáveis numéricas',
    labels=dict(color='Correlação')
)

fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='investimento_recebido',
    y='crescimento_percentual',
    color='setor',
    size='faturamento',
    hover_data=['startup', 'regiao', 'uf', 'estagio', 'tecnologia_principal', 'nivel_inovacao'],
    title='Relação entre investimento recebido e crescimento percentual',
    labels={
        'investimento_recebido': 'Investimento recebido (R$)',
        'crescimento_percentual': 'Crescimento percentual (%)',
        'setor': 'Setor'
    }
)

fig.update_layout(template='plotly_white')
fig.show()

### Interpretação

A dispersão investimento x crescimento ajuda a verificar se startups mais financiadas crescem mais. Caso a correlação seja fraca, isso indica que o investimento isolado não explica totalmente o crescimento. Nesse cenário, fatores como estágio da startup, tecnologia principal, setor, nível de inovação e capacidade de execução também devem ser considerados.

## 12. Tecnologias emergentes

In [ ]:
tecnologias = df.groupby('tecnologia_principal').agg(
    quantidade_registros=('startup', 'count'),
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean')
).reset_index().sort_values('quantidade_registros', ascending=False)

tecnologias

In [ ]:
fig = px.bar(
    tecnologias,
    x='tecnologia_principal',
    y='quantidade_registros',
    title='Quantidade de registros por tecnologia principal',
    labels={'tecnologia_principal': 'Tecnologia principal', 'quantidade_registros': 'Quantidade de registros'}
)

fig.update_layout(template='plotly_white', xaxis_tickangle=-35)
fig.show()

### Interpretação

A análise de tecnologias principais permite identificar tendências de inovação. Tecnologias como IA, cloud, blockchain, IoT, SaaS e Big Data representam áreas estratégicas para transformação digital. A tecnologia mais frequente pode indicar maior adoção pelo mercado, enquanto tecnologias com alto crescimento médio podem sinalizar expansão futura.

## 13. Heatmap de inovação

In [ ]:
heatmap_inovacao = pd.crosstab(df['setor'], df['nivel_inovacao'])
heatmap_inovacao

In [ ]:
fig = px.imshow(
    heatmap_inovacao,
    text_auto=True,
    aspect='auto',
    title='Heatmap de inovação por setor',
    labels=dict(x='Nível de inovação', y='Setor', color='Quantidade')
)

fig.update_layout(template='plotly_white')
fig.show()

### Interpretação

O heatmap permite enxergar a distribuição dos níveis de inovação dentro de cada setor. Setores com maior concentração de startups classificadas como Alto ou Disruptivo podem ser considerados mais relevantes para estratégias de transformação tecnológica.

## 14. Ranking de startups

In [ ]:
ranking_startups = df.groupby(['startup', 'setor', 'uf', 'regiao']).agg(
    investimento_total=('investimento_recebido', 'sum'),
    faturamento_total=('faturamento', 'sum'),
    crescimento_medio=('crescimento_percentual', 'mean'),
    funcionarios_medios=('funcionarios', 'mean')
).reset_index().sort_values('faturamento_total', ascending=False)

ranking_startups.head(20)

In [ ]:
fig = px.bar(
    ranking_startups.head(15),
    x='startup',
    y='faturamento_total',
    color='setor',
    title='Top 15 startups por faturamento acumulado',
    labels={'startup': 'Startup', 'faturamento_total': 'Faturamento total (R$)', 'setor': 'Setor'}
)

fig.update_layout(template='plotly_white', xaxis_tickangle=-35)
fig.show()

### Interpretação

O ranking de startups destaca empresas com maior desempenho econômico. Essa visão é útil para investidores, aceleradoras e gestores que desejam identificar empresas com maior potencial de escala e geração de receita.

## 15. Tabela dinâmica analítica

In [ ]:
tabela_dinamica = pd.pivot_table(
    df,
    values=['investimento_recebido', 'faturamento', 'crescimento_percentual', 'funcionarios'],
    index=['regiao', 'uf'],
    columns='setor',
    aggfunc={
        'investimento_recebido': 'sum',
        'faturamento': 'sum',
        'crescimento_percentual': 'mean',
        'funcionarios': 'mean'
    },
    fill_value=0
)

tabela_dinamica

## 16. Insights principais

1. O ecossistema possui diversidade setorial, com presença de fintechs, healthtechs, edtechs, agritechs, IA e retailtechs.
2. O investimento está distribuído entre setores, mas alguns concentram maior volume financeiro.
3. A comparação regional mostra concentração em determinados polos, o que sugere desigualdade na distribuição da inovação.
4. A relação entre investimento e crescimento deve ser interpretada com cautela, pois investimento não explica sozinho o desempenho.
5. Tecnologias emergentes são fundamentais para entender a direção da transformação digital.
6. Startups com maior faturamento podem representar maior maturidade ou melhor capacidade de monetização.
7. Níveis de inovação mais altos ajudam a identificar setores com maior potencial disruptivo.

## 17. Limitações da análise

- A base é simulada, portanto não representa necessariamente o mercado real brasileiro.
- Os dados têm estrutura mensal e podem conter repetições de startups ao longo do tempo.
- A análise mede associação, não causalidade.
- Investimento recebido não garante crescimento futuro.
- Outros fatores relevantes não estão presentes, como valuation, churn, lucro líquido, rodada específica, perfil dos fundadores e mercado-alvo.

## 18. Conclusão executiva

A análise indica que o ecossistema brasileiro de startups deve ser avaliado a partir de múltiplas dimensões: setor, região, tecnologia, investimento, faturamento, crescimento e nível de inovação.

A principal mensagem é que **não basta olhar apenas para quantidade de startups ou volume de investimento**. A melhor interpretação surge quando se combina investimento, crescimento, faturamento e grau de inovação.

Para um gestor, investidor ou formulador de política pública, a recomendação é priorizar setores e regiões que apresentem simultaneamente:
- alto investimento;
- crescimento consistente;
- boa geração de faturamento;
- presença de tecnologias emergentes;
- maior proporção de startups com inovação alta ou disruptiva.

Essa abordagem torna a decisão mais robusta e reduz o risco de investir apenas em setores populares, mas sem evidências claras de desempenho.

## 19. Próximos passos

Como evolução do projeto, recomenda-se:
- publicar o dashboard no Streamlit Cloud;
- publicar o `index.html` via GitHub Pages;
- adicionar persistência com SQLite;
- criar uma versão multipágina do dashboard;
- incluir dados reais de mercado para comparação;
- aplicar modelos preditivos para estimar crescimento futuro.